# Final Comparison Notebook — Module 3

Consolidates every prior result into the 4 comparisons needed for the
dissertation: **Baseline Comparison**, **Ablation Study**, **Drift Adaptation
Analysis**, and a rolled-up **Performance Analysis**.

**Three configurations of the proposed model are evaluated on every set**
(`cc1_test`, `drift_sc1`, `drift_sc2`, `drift_cc2`):
1. **VAE alone** — reconstruction error, static `val_p99` threshold (no adaptation).
2. **VAE + Adaptive Threshold** — the validated blended/shrinkage threshold.
3. **Full Model** — adaptive threshold **+** drift-triggered incremental
   learning (KS-test against the CC1-train baseline, small SGD fine-tune on
   detected drift).

Configuration 3 was previously only run end-to-end on `drift_cc2`. **This
notebook runs it on all four sets**, so the ablation and drift-adaptation
tables are complete rather than partial.

**A conceptual point made explicit here, not just asserted:** thresholding
alone (config 1 → 2) cannot change PR-AUC/ROC-AUC — those are computed from the
raw reconstruction-error scores, and a threshold only decides where to cut that
same ranking. Only incremental learning (config 2 → 3) can move PR-AUC/ROC-AUC,
because it's the only step that changes the scores themselves. The tables below
demonstrate this directly rather than just stating it.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import pickle, joblib, os, time
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, confusion_matrix,
)
from collections import deque

BASE      = r'c:\Users\DELL\Documents\Claude\Projects\FYP\module3'
DATA_DIR  = os.path.join(BASE, 'data', 'processed')
WIN_DIR   = os.path.join(BASE, 'data', 'processed', 'windows_cc1')
MODEL_DIR = os.path.join(BASE, 'models')

WINDOW_SIZE = 30
FEATURE_COLS = [
    'container_cpu_usage_seconds_rate', 'container_cpu_system_seconds_rate', 'container_cpu_user_seconds_rate',
    'container_memory_usage_bytes', 'container_memory_working_set_bytes', 'container_memory_rss', 'container_memory_cache',
]
SPLIT_FILES = {
    'cc1_test':  'cc1_test.csv',
    'drift_cc2': 'drift_complex_case2.csv',
}
DRIFT_SETS = ['drift_cc2']
ALL_SETS   = ['cc1_test'] + DRIFT_SETS
EVAL_SETS  = ALL_SETS

K_ADAPTIVE, BUFFER_SIZE, PRIOR_STRENGTH = 3.0, 500, 500
REFIT_INTERVAL, FT_BUFFER_SIZE, FT_LR, FT_EPOCHS, KS_ALPHA = 5000, 2000, 1e-4, 5, 0.001

print('Paths and constants configured.')

# Stage E scope
DRIFT_SETS = ['drift_cc2']
EVAL_SETS = ['cc1_test', 'drift_cc2']
# Stage B/C knobs (mirrored from adaptive / incremental notebooks)
REL_SCORE_EPS = 1e-8
KS_STREAK_REQUIRED = 2
UPDATE_REFERENCE = True
REF_EMA_ALPHA = 0.5
FT_PURIFY_QUANTILE = 0.80
USE_RELATIVE_DECISION = True
K_REL = 3.0
print('final_comparison Stage B/C knobs ready.')


## Step 1 — Load model + reference baseline

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden1, hidden2, latent_dim):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden1), nn.ReLU(), nn.Linear(hidden1, hidden2), nn.ReLU())
        self.fc_mu = nn.Linear(hidden2, latent_dim)
        self.fc_lv = nn.Linear(hidden2, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim, hidden2), nn.ReLU(), nn.Linear(hidden2, hidden1), nn.ReLU(), nn.Linear(hidden1, input_dim))
    def encode(self, x):
        h = self.encoder(x); return self.fc_mu(h), torch.clamp(self.fc_lv(h), -10, 10)
    def decode(self, z): return self.decoder(z)
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar); return mu + torch.randn_like(std) * std
    def forward(self, x):
        mu, logvar = self.encode(x); z = self.reparameterize(mu, logvar); return self.decode(z), mu, logvar
    @torch.no_grad()
    def anomaly_score(self, x):
        self.eval(); mu, _ = self.encode(x); return ((self.decode(mu) - x) ** 2).mean(dim=1)

meta        = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_meta.pkl'), 'rb'))
static_eval = pickle.load(open(os.path.join(MODEL_DIR, 'vae_cc1_eval.pkl'), 'rb'))
CLIP          = meta['clip']
GLOBAL_MEAN   = meta['mu_train']
GLOBAL_STD    = meta['sigma_train']
STATIC_THRESH = static_eval['thresholds']['val_p99']

base_model = VAE(meta['input_dim'], meta['hidden1'], meta['hidden2'], meta['latent_dim'])
base_model.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'vae_cc1.pt'), map_location='cpu'))
base_model.eval()

X_cc1_train = np.clip(np.load(os.path.join(WIN_DIR, 'X_cc1_train.npy')), -CLIP, CLIP).astype(np.float32)
rng = np.random.default_rng(42)
REF_SAMPLE = X_cc1_train[rng.choice(len(X_cc1_train), size=5000, replace=False)]
with torch.no_grad():
    REFERENCE_MSE = base_model.anomaly_score(torch.from_numpy(REF_SAMPLE)).numpy()

print(f'Static threshold (val_p99): {STATIC_THRESH:.5f}')
print(f'Model loaded. Fixed KS-test reference: {len(REFERENCE_MSE):,} cc1_train windows.')

_blended_path = os.path.join(MODEL_DIR, 'vae_cc1_adaptive_blended_eval.pkl')
if os.path.exists(_blended_path):
    _bl = pickle.load(open(_blended_path, 'rb'))
    if 'best_k_rel' in _bl:
        K_REL = float(_bl['best_k_rel'])
        print(f'Loaded K_REL={K_REL} from Stage B blended eval')
    if 'prior_strength' in _bl:
        PRIOR_STRENGTH = int(_bl['prior_strength'])


## Step 2 — Load all 4 sets in true chronological order (with cmdb_id + PCA features)

Same replay-and-verify approach used throughout this project: rebuild from the
raw split CSVs, verify against the already-saved, already-verified `.npy`
arrays before trusting anything downstream.

In [3]:
def window_meta_with_features(df, feature_cols, window_size=WINDOW_SIZE, stride=1):
    X, cmdb_ids, end_ts, ys, fts = [], [], [], [], []
    for cmdb_id, g in df.sort_values('timestamp').groupby('cmdb_id'):
        data = g[feature_cols].values.astype(np.float32)
        is_gap = g['is_gap'].values
        labels = g['label'].values
        ftypes = g['failure_type'].values.astype(object)
        ts     = g['timestamp'].values
        n = len(g)
        for i in range(0, n - window_size + 1, stride):
            if is_gap[i:i + window_size].any():
                continue
            X.append(data[i:i + window_size])
            cmdb_ids.append(cmdb_id)
            end_ts.append(ts[i + window_size - 1])
            ys.append(int(labels[i:i + window_size].any()))
            w_types = sorted({t for t in ftypes[i:i + window_size] if isinstance(t, str)})
            fts.append(','.join(w_types) if w_types else None)
    return np.stack(X), np.array(cmdb_ids), np.array(end_ts), np.array(ys, dtype=np.int64), np.array(fts, dtype=object)

pca_bundle = joblib.load(os.path.join(MODEL_DIR, 'cc1_pca.pkl'))
pca = pca_bundle['pca']

streams = {}
for name, fname in SPLIT_FILES.items():
    df = pd.read_csv(os.path.join(DATA_DIR, fname), low_memory=False)
    df['is_gap'] = df['is_gap'].astype(bool)
    X_raw, cmdb_ids, end_ts, y_true, ft_true = window_meta_with_features(df, FEATURE_COLS)

    y_saved = np.load(os.path.join(WIN_DIR, f'y_{name}.npy'))
    match = np.array_equal(y_true, y_saved)
    assert match, f'{name}: rebuild does not match saved arrays.'

    X_flat = X_raw.reshape(len(X_raw), -1)
    X_pca = np.clip(pca.transform(X_flat), -CLIP, CLIP).astype(np.float32)

    order = np.argsort(end_ts, kind='stable')
    streams[name] = {
        'X': X_pca[order], 'y': y_true[order], 'ft': ft_true[order], 'cmdb': cmdb_ids[order], 'ts': end_ts[order],
    }
    print(f'  {name:10s}: {len(y_true):>7,} windows  |  order/label check: OK')


  cc1_test  :  44,185 windows  |  order/label check: OK
  drift_sc1 :  59,697 windows  |  order/label check: OK
  drift_sc2 :  38,070 windows  |  order/label check: OK
  drift_cc2 :  76,977 windows  |  order/label check: OK


## Section 1 — Baseline Comparison (Gaussian vs. Isolation Forest vs. Standard VAE)

Same features (whitened PCA), same leak-free threshold-selection discipline
(calibrated only from `cc1_val`'s false-positive rate).

In [ ]:
X_val = np.clip(np.load(os.path.join(WIN_DIR, 'X_cc1_val.npy')), -CLIP, CLIP).astype(np.float32)
X_train_full = np.clip(np.load(os.path.join(WIN_DIR, 'X_cc1_train.npy')), -CLIP, CLIP).astype(np.float32)

def gaussian_score(Xarr):
    return (Xarr ** 2).sum(axis=1)

iso_forest = IsolationForest(n_estimators=100, contamination='auto', random_state=42, n_jobs=-1)
iso_forest.fit(X_train_full)

gauss_val = gaussian_score(X_val)
iso_val   = -iso_forest.decision_function(X_val)
with torch.no_grad():
    vae_val = base_model.anomaly_score(torch.from_numpy(X_val)).numpy()

thresh_gauss = float(np.percentile(gauss_val, 99))
thresh_iso   = float(np.percentile(iso_val, 99))
print(f'Gaussian val_p99: {thresh_gauss:.4f}   Isolation Forest val_p99: {thresh_iso:.4f}   VAE val_p99: {STATIC_THRESH:.5f}')

def evaluate(scores, y_true, threshold):
    auc_roc = roc_auc_score(y_true, scores)
    auc_pr  = average_precision_score(y_true, scores)
    pred = (scores > threshold).astype(int)
    return {
        'auc_roc': float(auc_roc), 'auc_pr': float(auc_pr),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall':    float(recall_score(y_true, pred, zero_division=0)),
        'f1':        float(f1_score(y_true, pred, zero_division=0)),
    }

# --- VAE + Gaussian fusion (α=0.5), train-only z-norm ---
DEPLOY_ALPHA = 0.5
with torch.no_grad():
    vae_train = base_model.anomaly_score(torch.from_numpy(X_train_full)).numpy()
gauss_train = gaussian_score(X_train_full)
mu_v, sig_v = float(vae_train.mean()), max(float(vae_train.std()), 1e-8)
mu_g, sig_g = float(gauss_train.mean()), max(float(gauss_train.std()), 1e-8)

def fused_score(Xs):
    with torch.no_grad():
        sv = base_model.anomaly_score(torch.from_numpy(Xs)).numpy()
    sg = gaussian_score(Xs)
    zv = (sv - mu_v) / sig_v
    zg = (sg - mu_g) / sig_g
    return DEPLOY_ALPHA * zv + (1.0 - DEPLOY_ALPHA) * zg, sv, sg

fuse_val = DEPLOY_ALPHA * ((vae_val - mu_v) / sig_v) + (1.0 - DEPLOY_ALPHA) * ((gauss_val - mu_g) / sig_g)
thresh_fuse = float(np.percentile(fuse_val, 99))
print(f'Fusion α={DEPLOY_ALPHA} val_p99: {thresh_fuse:.4f}')

baseline_results = {'gaussian': {}, 'isolation_forest': {}, 'vae_alone': {}, 'fusion_a0.5': {}}
for name in ALL_SETS:
    Xs = streams[name]['X']; ys = streams[name]['y']
    baseline_results['gaussian'][name]         = evaluate(gaussian_score(Xs), ys, thresh_gauss)
    baseline_results['isolation_forest'][name] = evaluate(-iso_forest.decision_function(Xs), ys, thresh_iso)
    fuse_s, mse, _ = fused_score(Xs)
    baseline_results['vae_alone'][name] = evaluate(mse, ys, STATIC_THRESH)
    baseline_results['fusion_a0.5'][name] = evaluate(fuse_s, ys, thresh_fuse)

print(f'\n{"set":12s} {"method":18s} {"PR-AUC":>8s} {"ROC-AUC":>9s} {"F1":>7s} {"Precision":>10s} {"Recall":>8s}')
for name in ALL_SETS:
    for method in ['gaussian', 'isolation_forest', 'vae_alone', 'fusion_a0.5']:
        r = baseline_results[method][name]
        print(f'{name:12s} {method:18s} {r["auc_pr"]:8.4f} {r["auc_roc"]:9.4f} {r["f1"]:7.3f} {r["precision"]:10.3f} {r["recall"]:8.3f}')
    print()


## Section 2 — Ablation Study

`VAE alone` → `VAE + Adaptive Threshold` → `Full Model` (+ incremental
learning), run on **all four** sets. `run_stream` reproduces
`adaptive_threshold_blended.ipynb`'s validated blended threshold when
`incremental_learning=False`, and adds the drift-triggered SGD fine-tune from
`incremental_learning.ipynb` when `True`.

In [ ]:
from collections import deque
from scipy import stats
import copy

def run_stream(model, X_stream, cmdb_stream, incremental_learning, reference_mse,
               verbose=False, update_reference=UPDATE_REFERENCE,
               use_relative=USE_RELATIVE_DECISION, k_rel=None):
    """Stage B/C stream used by ablation: relative decision + stabilized FT."""
    if k_rel is None:
        k_rel = K_REL
    model = copy.deepcopy(model)
    threshold_buffers = {}
    ft_pool = deque(maxlen=FT_BUFFER_SIZE)
    working_ref = np.asarray(reference_mse, dtype=np.float64).copy()
    preds, mse_trace, rel_trace = [], [], []
    n_finetunes, finetune_events, drift_streak = 0, [], 0
    opt = torch.optim.Adam(model.parameters(), lr=FT_LR) if incremental_learning else None

    for i in range(len(X_stream)):
        x_i = X_stream[i]
        cid = cmdb_stream[i]
        with torch.no_grad():
            model.eval()
            mse_i = float(model.anomaly_score(torch.from_numpy(x_i[None, :])).item())

        buf = threshold_buffers.setdefault(cid, deque(maxlen=BUFFER_SIZE))
        n_local = len(buf)
        w = n_local / (n_local + PRIOR_STRENGTH)
        if n_local == 0:
            lm, ls = GLOBAL_MEAN, GLOBAL_STD
        else:
            arr = np.fromiter(buf, dtype=np.float64)
            lm, ls = float(arr.mean()), float(arr.std())
            if ls < REL_SCORE_EPS:
                ls = GLOBAL_STD
        bm = w * lm + (1 - w) * GLOBAL_MEAN
        bs = max(w * ls + (1 - w) * GLOBAL_STD, REL_SCORE_EPS)
        rel = (mse_i - bm) / (bs + REL_SCORE_EPS)

        if use_relative:
            is_anom = rel > k_rel
        else:
            t = bm + K_ADAPTIVE * bs
            is_anom = mse_i > t

        preds.append(bool(is_anom))
        mse_trace.append(mse_i)
        rel_trace.append(rel)

        if not is_anom:
            buf.append(mse_i)
            ft_pool.append((x_i, mse_i, rel))

        if incremental_learning and ((i + 1) % REFIT_INTERVAL == 0) and len(ft_pool) >= FT_BUFFER_SIZE // 2:
            rels = np.array([r for _, _, r in ft_pool], dtype=np.float64)
            cutoff = float(np.quantile(rels, FT_PURIFY_QUANTILE))
            purified = [(x, m, r) for (x, m, r) in ft_pool if r <= cutoff]
            if len(purified) < max(100, FT_BUFFER_SIZE // 4):
                purified = list(ft_pool)
            X_ft = np.stack([x for x, _, _ in purified]).astype(np.float32)
            with torch.no_grad():
                model.eval()
                recent_mse = model.anomaly_score(torch.from_numpy(X_ft)).numpy()
            ks_stat, p_val = stats.ks_2samp(working_ref, recent_mse)
            drifted = bool(p_val < KS_ALPHA)
            drift_streak = drift_streak + 1 if drifted else 0
            if drifted and drift_streak >= KS_STREAK_REQUIRED:
                model.train()
                X_t = torch.from_numpy(X_ft)
                for _ep in range(FT_EPOCHS):
                    perm = torch.randperm(len(X_t))
                    for start in range(0, len(X_t), 512):
                        batch = X_t[perm[start:start + 512]]
                        opt.zero_grad()
                        mu, lv = model.encode(batch)
                        recon = model.decode(mu)
                        recon_loss = ((recon - batch) ** 2).mean()
                        kl = -0.5 * torch.mean(torch.sum(1 + lv - mu.pow(2) - lv.exp(), dim=1))
                        (recon_loss + 0.01 * kl).backward()
                        opt.step()
                n_finetunes += 1
                finetune_events.append(i)
                drift_streak = 0
                if update_reference:
                    with torch.no_grad():
                        model.eval()
                        post_mse = model.anomaly_score(torch.from_numpy(X_ft)).numpy()
                    rng = np.random.default_rng(42 + n_finetunes)
                    if len(post_mse) >= len(working_ref):
                        new_sample = rng.choice(post_mse, size=len(working_ref), replace=False)
                    else:
                        new_sample = rng.choice(post_mse, size=len(working_ref), replace=True)
                    working_ref = REF_EMA_ALPHA * new_sample + (1.0 - REF_EMA_ALPHA) * working_ref
                if verbose:
                    print(f'  FT at window {i+1}: KS={ks_stat:.4f} p={p_val:.2e} n_ft={n_finetunes}')

    model.eval()
    return np.array(preds), np.array(mse_trace), np.array(rel_trace), n_finetunes, finetune_events

print('final_comparison run_stream updated (Stage B/C).')


In [ ]:
ablation_raw = {}
for name in ALL_SETS:
    print(f'--- {name} ---')
    Xs, ys, cmdbs = streams[name]['X'], streams[name]['y'], streams[name]['cmdb']

    preds_adapt, mse_adapt, rel_adapt, _, _ = run_stream(base_model, Xs, cmdbs, incremental_learning=False, reference_mse=REFERENCE_MSE)
    preds_full,  mse_full,  rel_full, nft, events = run_stream(base_model, Xs, cmdbs, incremental_learning=True, reference_mse=REFERENCE_MSE, verbose=True)

    ablation_raw[name] = {
        'preds_adapt': preds_adapt, 'mse_adapt': mse_adapt,
        'preds_full': preds_full, 'mse_full': mse_full,
        'n_finetunes': nft, 'finetune_events': events,
    }
    print(f'  fine-tune events: {nft}\n')


In [7]:
ablation_results = {'vae_alone': {}, 'vae_adaptive': {}, 'full_model': {}}
print(f'{"set":12s} {"config":16s} {"PR-AUC":>8s} {"ROC-AUC":>9s} {"F1":>7s} {"Precision":>10s} {"Recall":>8s}')
for name in ALL_SETS:
    ys = streams[name]['y']
    r = ablation_raw[name]

    # VAE alone: reuse the static-threshold scores computed in Section 1 (identical model, no adaptation at all)
    ablation_results['vae_alone'][name] = baseline_results['vae_alone'][name]

    # VAE + Adaptive: SAME underlying scores as VAE alone (thresholding cannot change ranking) —
    # only the classification decision changes, so AUC figures are copied, not recomputed.
    with torch.no_grad():
        mse_static = base_model.anomaly_score(torch.from_numpy(streams[name]['X'])).numpy()
    ablation_results['vae_adaptive'][name] = {
        'auc_roc': roc_auc_score(ys, mse_static), 'auc_pr': average_precision_score(ys, mse_static),
        'precision': precision_score(ys, r['preds_adapt'], zero_division=0),
        'recall':    recall_score(ys, r['preds_adapt'], zero_division=0),
        'f1':        f1_score(ys, r['preds_adapt'], zero_division=0),
    }

    # Full Model: scores THEMSELVES changed (incremental learning), so AUC is recomputed on the updated trace.
    ablation_results['full_model'][name] = {
        'auc_roc': roc_auc_score(ys, r['mse_full']), 'auc_pr': average_precision_score(ys, r['mse_full']),
        'precision': precision_score(ys, r['preds_full'], zero_division=0),
        'recall':    recall_score(ys, r['preds_full'], zero_division=0),
        'f1':        f1_score(ys, r['preds_full'], zero_division=0),
    }

    for config in ['vae_alone', 'vae_adaptive', 'full_model']:
        m = ablation_results[config][name]
        print(f'{name:12s} {config:16s} {m["auc_pr"]:8.4f} {m["auc_roc"]:9.4f} {m["f1"]:7.3f} {m["precision"]:10.3f} {m["recall"]:8.3f}')
    print()


set          config             PR-AUC   ROC-AUC      F1  Precision   Recall
cc1_test     vae_alone          0.6014    0.8763   0.618      0.630    0.605
cc1_test     vae_adaptive       0.6014    0.8763   0.623      0.676    0.578
cc1_test     full_model         0.6052    0.8851   0.634      0.701    0.578

drift_sc1    vae_alone          0.0181    0.8515   0.040      0.020    0.806
drift_sc1    vae_adaptive       0.0181    0.8515   0.047      0.024    0.813
drift_sc1    full_model         0.0198    0.8736   0.052      0.027    0.806

drift_sc2    vae_alone          0.0615    0.6327   0.024      0.012    0.482
drift_sc2    vae_adaptive       0.0615    0.6327   0.039      0.020    0.494
drift_sc2    full_model         0.0578    0.6982   0.043      0.022    0.494

drift_cc2    vae_alone          0.4089    0.8812   0.276      0.168    0.772
drift_cc2    vae_adaptive       0.4089    0.8812   0.347      0.223    0.786
drift_cc2    full_model         0.4218    0.9020   0.391      0.262    0.

## Section 3 — Drift Adaptation Analysis

- **Before drift**: `VAE alone` on `cc1_test` (in-distribution, no drift present).
- **After drift**: `VAE alone` on each drift set (shows the raw performance drop).
- **After adaptation**: `Full Model` on each drift set (shows how much of that drop is recovered).

**Performance retention** = drift-set metric ÷ in-distribution metric — a
single number that directly operationalizes "drift-aware": closer to 1.0 means
the system held onto more of its in-distribution capability under drift.

In [8]:
before_drift = ablation_results['vae_alone']['cc1_test']
print(f'{"metric":10s} {"before drift":>14s}')
for metric in ['auc_pr', 'f1', 'recall']:
    print(f'{metric:10s} {before_drift[metric]:14.4f}')

print(f'\n{"drift set":12s} {"metric":8s} {"after drift":>12s} {"after adaptation":>17s} {"retention (drift)":>19s} {"retention (adapted)":>21s}')
drift_analysis = {}
for name in DRIFT_SETS:
    after_drift = ablation_results['vae_alone'][name]
    after_adapt = ablation_results['full_model'][name]
    drift_analysis[name] = {}
    for metric in ['auc_pr', 'f1', 'recall']:
        ret_drift  = after_drift[metric] / before_drift[metric] if before_drift[metric] > 0 else float('nan')
        ret_adapt  = after_adapt[metric] / before_drift[metric] if before_drift[metric] > 0 else float('nan')
        drift_analysis[name][metric] = {'after_drift': after_drift[metric], 'after_adaptation': after_adapt[metric],
                                          'retention_drift': ret_drift, 'retention_adapted': ret_adapt}
        print(f'{name:12s} {metric:8s} {after_drift[metric]:12.4f} {after_adapt[metric]:17.4f} {ret_drift:19.2%} {ret_adapt:21.2%}')


metric       before drift
auc_pr             0.6014
f1                 0.6175
recall             0.6055

drift set    metric    after drift  after adaptation   retention (drift)   retention (adapted)
drift_sc1    auc_pr         0.0181            0.0198               3.01%                 3.30%
drift_sc1    f1             0.0399            0.0518               6.46%                 8.39%
drift_sc1    recall         0.8060            0.8060             133.12%               133.12%
drift_sc2    auc_pr         0.0615            0.0578              10.23%                 9.60%
drift_sc2    f1             0.0235            0.0429               3.81%                 6.95%
drift_sc2    recall         0.4824            0.4941              79.67%                81.61%
drift_cc2    auc_pr         0.4089            0.4218              68.00%                70.13%
drift_cc2    f1             0.2757            0.3910              44.64%                63.31%
drift_cc2    recall         0.7722      

## Section 4 — Performance Analysis Summary (PR-AUC, F1, Recall)

Rolls up Sections 1–2 into the three headline metrics, all methods, all sets.

In [ ]:
summary_methods = ['gaussian', 'isolation_forest', 'vae_alone', 'fusion_a0.5', 'vae_adaptive', 'full_model']
summary_sources = {**baseline_results, **{k: v for k, v in ablation_results.items() if k not in baseline_results}}

print(f'{"set":12s} {"method":16s} {"PR-AUC":>8s} {"F1":>7s} {"Recall":>8s}')
for name in ALL_SETS:
    for method in summary_methods:
        m = summary_sources[method][name]
        print(f'{name:12s} {method:16s} {m["auc_pr"]:8.4f} {m["f1"]:7.3f} {m["recall"]:8.3f}')
    print()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x = np.arange(len(ALL_SETS))
width = 0.15
for i, method in enumerate(summary_methods):
    for ax, metric, title in zip(axes, ['auc_pr', 'f1', 'recall'], ['PR-AUC', 'F1', 'Recall']):
        vals = [summary_sources[method][name][metric] for name in ALL_SETS]
        ax.bar(x + i * width, vals, width, label=method)
for ax, title in zip(axes, ['PR-AUC', 'F1', 'Recall']):
    ax.set_xticks(x + width * 2); ax.set_xticklabels(ALL_SETS, rotation=20)
    ax.set_title(title); ax.legend(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'final_comparison_summary.png'), dpi=120)
plt.show()


## Save all results

In [10]:
save_results = {
    'baseline_comparison': baseline_results,
    'ablation_study': ablation_results,
    'drift_adaptation_analysis': {'before_drift': before_drift, 'per_drift_set': drift_analysis},
    'finetune_events': {name: ablation_raw[name]['finetune_events'] for name in ALL_SETS},
}
out_path = os.path.join(MODEL_DIR, 'final_comparison_results.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(save_results, f)
print(f'Saved -> {out_path}')
print(f'Saved -> {os.path.join(MODEL_DIR, "final_comparison_summary.png")}')


Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\models\final_comparison_results.pkl
Saved -> c:\Users\jthar\Documents\Claude\Projects\module3\models\final_comparison_summary.png


## How to read this notebook's output

- **Section 1** answers: does the proposed VAE beat simple baselines? (Answer
  varies by set — see the printed table.)
- **Section 2** proves the conceptual point from the intro: `vae_alone` and
  `vae_adaptive`'s PR-AUC/ROC-AUC should be **identical** per set (thresholding
  doesn't change ranking) — only F1/precision/recall differ. `full_model`'s
  PR-AUC/ROC-AUC can differ from both, since incremental learning changes the
  underlying scores. If `n_finetunes=0` for a set (check the per-set printout
  in the ablation cell), incremental learning correctly did nothing there —
  most likely `cc1_test`, since there's no real drift to detect.
- **Section 3** is the single clearest table for the "drift-aware" claim:
  retention closer to 100% after adaptation than after drift-alone is direct,
  quantified evidence of drift adaptation working.
- **Section 4** is the rollup table/figure for the results chapter.

## Stage E — Consolidated improvement rollup (relative + Stage C)

Loads Stage B (`vae_cc1_adaptive_blended_eval.pkl`) and Stage C (`incremental_learning_eval.pkl`) results and compares against the plan target table. Also records whether Stage D (window=60) is required.


In [ ]:

# Stage E consolidation — read Stage B/C artifacts and decide Stage D
from pathlib import Path

targets = {
    'cc1_test': {
        'auc_pr': 0.63, 'f1': 0.63, 'precision': 0.65, 'recall': 0.55, 'fpr': 0.01,
    },
    'drift_cc2': {
        'auc_pr': 0.48, 'f1': 0.45, 'precision': 0.35, 'recall': 0.65, 'fpr': 0.015,
    },
}

blended_path = os.path.join(MODEL_DIR, 'vae_cc1_adaptive_blended_eval.pkl')
incr_path = os.path.join(MODEL_DIR, 'incremental_learning_eval.pkl')
static_path = os.path.join(MODEL_DIR, 'vae_cc1_eval.pkl')

rollup = {'static': {}, 'relative': {}, 'full_model': {}, 'targets': targets}

if os.path.exists(static_path):
    st = pickle.load(open(static_path, 'rb'))
    for name in ['cc1_test', 'drift_cc2']:
        if name in st.get('auc', {}):
            pr = st['precision_recall'][name]['val_p99']
            rollup['static'][name] = {
                'auc_pr': st['auc'][name]['auc_pr'],
                'auc_roc': st['auc'][name]['auc_roc'],
                'f1': pr['f1'], 'precision': pr['precision'], 'recall': pr['recall'],
                'fpr': pr.get('fpr'),
            }

if os.path.exists(blended_path):
    bl = pickle.load(open(blended_path, 'rb'))
    rel = bl.get('relative_results', {})
    for name, r in rel.items():
        rollup['relative'][name] = {k: r[k] for k in
            ['auc_pr', 'auc_roc', 'f1', 'precision', 'recall', 'fpr'] if k in r}
    rollup['stage_b_pass'] = bl.get('stage_b_pass')
    rollup['best_k_rel'] = bl.get('best_k_rel')

if os.path.exists(incr_path):
    inc = pickle.load(open(incr_path, 'rb'))
    rollup['full_model']['drift_cc2'] = inc.get('treatment', {})
    rollup['full_model']['control_drift_cc2'] = inc.get('control', {})
    rollup['n_finetunes_treatment'] = inc.get('n_finetunes_treatment')
    rollup['finetune_events'] = inc.get('finetune_events')

print('=== Improvement rollup vs targets ===')
print(f'{"set":12s} {"config":12s} {"PR-AUC":>8s} {"F1":>7s} {"Prec":>7s} {"Rec":>7s} {"FPR":>8s}')
for name in ['cc1_test', 'drift_cc2']:
    for cfg in ['static', 'relative', 'full_model']:
        r = rollup.get(cfg, {}).get(name)
        if not r:
            continue
        print(f'{name:12s} {cfg:12s} {r.get("auc_pr", float("nan")):8.4f} '
              f'{r.get("f1", float("nan")):7.3f} {r.get("precision", float("nan")):7.3f} '
              f'{r.get("recall", float("nan")):7.3f} {r.get("fpr", float("nan")):8.4f}')

# Stage D trigger: ID PR-AUC still < 0.65 after relative/full
id_prauc_candidates = []
if 'cc1_test' in rollup['relative']:
    id_prauc_candidates.append(rollup['relative']['cc1_test'].get('auc_pr', 0))
if 'cc1_test' in rollup['static']:
    id_prauc_candidates.append(rollup['static']['cc1_test'].get('auc_pr', 0))
best_id = max(id_prauc_candidates) if id_prauc_candidates else 0.0
stage_d_needed = bool(best_id < 0.65)
rollup['stage_d_needed'] = stage_d_needed
rollup['best_id_pr_auc'] = best_id
print(f'\nBest ID PR-AUC observed in rollup: {best_id:.4f}')
print(f'Stage D (window=60) needed per plan rule (<0.65)? {stage_d_needed}')
print('NOTE: Stage D is LAST RESORT and must also satisfy joint drift criteria; '
      'prior experiments showed w=60 hurts drift. Default remains w=30.')

out_path = os.path.join(MODEL_DIR, 'stage_e_improvement_rollup.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(rollup, f)
print(f'Saved -> {out_path}')


## Stage D — Window-size last-resort decision

Plan rule: consider `WINDOW_SIZE=60` **only if** post–Stage B/C ID PR-AUC remains `< 0.65`,
and accept only if joint bar holds:
`ID_PR-AUC ≥ 0.75` AND `Drift_PR-AUC ≥ 0.38` AND `Drift_F1 ≥ 0.30` with Precision ≥ 0.25.

Prior measured evidence (experiments/, diagnosis only — not edited):
window=60 VAE-alone ID PR-AUC ≈ 0.92 but Drift PR-AUC ≈ 0.33 / F1 ≈ 0.16;
Full Model drift F1 ≈ 0.165. Joint bar fails → **keep WINDOW_SIZE=30**.


In [ ]:
# Stage D decision (does NOT change windowing_pca / train_vae unless accepted)
PRIOR_W60_EVIDENCE = {
    # from experiments/window_size_ablation + window_size_60_threshold_fix (read-only evidence)
    'cc1_test':  {'auc_pr': 0.9178, 'f1': 0.912, 'precision': 0.986, 'recall': 0.848},
    'drift_cc2': {'auc_pr': 0.3307, 'f1': 0.155, 'precision': 0.086, 'recall': 0.800},
    'full_model_drift_f1': 0.165,
}
JOINT = {
    'id_prauc_min': 0.75,
    'drift_prauc_min': 0.38,
    'drift_f1_min': 0.30,
    'drift_precision_min': 0.25,
}

# Prefer live rollup if Stage E already ran
best_id = None
if 'rollup' in dir() and isinstance(rollup, dict):
    best_id = rollup.get('best_id_pr_auc')
if best_id is None:
    # fall back to static eval on disk
    st_path = os.path.join(MODEL_DIR, 'vae_cc1_eval.pkl')
    if os.path.exists(st_path):
        st = pickle.load(open(st_path, 'rb'))
        best_id = st.get('auc', {}).get('cc1_test', {}).get('auc_pr', 0.601)
    else:
        best_id = 0.601

stage_d_triggered = bool(best_id < 0.65)
joint_ok = (
    PRIOR_W60_EVIDENCE['cc1_test']['auc_pr'] >= JOINT['id_prauc_min']
    and PRIOR_W60_EVIDENCE['drift_cc2']['auc_pr'] >= JOINT['drift_prauc_min']
    and PRIOR_W60_EVIDENCE['drift_cc2']['f1'] >= JOINT['drift_f1_min']
    and PRIOR_W60_EVIDENCE['drift_cc2']['precision'] >= JOINT['drift_precision_min']
)
# Even if triggered by ID PR-AUC, reject adoption when joint bar fails
adopt_window_60 = bool(stage_d_triggered and joint_ok)
decision = {
    'best_id_pr_auc': float(best_id),
    'stage_d_triggered': stage_d_triggered,
    'joint_bar_ok': joint_ok,
    'adopt_window_60': adopt_window_60,
    'final_window_size': 60 if adopt_window_60 else 30,
    'rationale': (
        'Adopt window=60' if adopt_window_60 else
        'Keep window=30 — Stage D triggered but joint drift bar fails on prior w=60 evidence'
        if stage_d_triggered else
        'Keep window=30 — ID PR-AUC already >= 0.65 after Stage B/C'
    ),
    'prior_w60_evidence': PRIOR_W60_EVIDENCE,
    'joint_criteria': JOINT,
}
print('=== Stage D decision ===')
for k, v in decision.items():
    if k == 'prior_w60_evidence':
        continue
    print(f'  {k}: {v}')

out_path = os.path.join(MODEL_DIR, 'stage_d_window_decision.pkl')
with open(out_path, 'wb') as f:
    pickle.dump(decision, f)
print(f'Saved -> {out_path}')
print('NO edits to windowing_pca.ipynb / train_vae.ipynb (window remains 30).')
